In [1]:
import pandas as pd
df = pd.read_csv("messy_sales.csv")
df.head(10)

,order_id,date,product,price,qty,zip
0,1254,04/06/2026,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,05/06/2026,mug,5.50,1,10001
3,1066,05/30/2026,notebook,NaN,1,98101
4,1114,04/06/2026,webcam,45.59,4,90405
5,1280,04/22/2026,pen set,NaN,4,30303
6,1204,04/18/2026,charger,8.41,2,2134
7,1249,2026-05-15,keyboard,75.04,4,90405
8,1037,2026-05-19,mug,9.31,2,60614
9,1177,2026-05-13,mug,10.11,3,90210


In [2]:
df.shape

(300, 6)

In [3]:
df.dtypes


order_id      int64
date            str
product         str
price       float64
qty           int64
zip           int64
dtype: object

### Problems I can see with this dataset are:
- date should be set as datetime so all the dates follow the same format
- zipcode should be set as string to prevent the first 0 from being taken out of the zipcodes starting with 0
- some of the price cells are left empty


In [4]:
## Checking how many null values there are
df.isna().sum()

order_id     0
date         0
product      0
price       12
qty          0
zip          0
dtype: int64

In [5]:
## creating a fill value variable
fill_value=df["price"].median()
## filling the null values with the median value variable I created
df["price"]=df["price"].fillna(fill_value)

print(f"fill value used is {fill_value}")


fill value used is 37.53


In [6]:
## verifying that there are no longer any null values
df.isna().sum()

order_id    0
date        0
product     0
price       0
qty         0
zip         0
dtype: int64

In [7]:
df.shape

(300, 6)

In [8]:
print(df.duplicated().sum())

8


In [9]:
## dropping duplicates
df = df.drop_duplicates()
## verifying duplicates were dropped
df.shape

(292, 6)

In [10]:
## changing zip into a string
df["zip"] = df["zip"].astype(str).str.zfill(5)
##verifying that all zips now have 5 characters
df.head(10)

,order_id,date,product,price,qty,zip
0,1254,04/06/2026,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,05/06/2026,mug,5.50,1,10001
3,1066,05/30/2026,notebook,37.53,1,98101
4,1114,04/06/2026,webcam,45.59,4,90405
5,1280,04/22/2026,pen set,37.53,4,30303
6,1204,04/18/2026,charger,8.41,2,02134
7,1249,2026-05-15,keyboard,75.04,4,90405
8,1037,2026-05-19,mug,9.31,2,60614
9,1177,2026-05-13,mug,10.11,3,90210


In [11]:
## changing date to datetime
df["date"] = pd.to_datetime(df["date"], format="mixed")
##verifying date is now unified
df.head(10)

,order_id,date,product,price,qty,zip
0,1254,2026-04-06,mug,11.36,4,60614
1,1057,2026-05-30,charger,14.79,3,30303
2,1150,2026-05-06,mug,5.50,1,10001
3,1066,2026-05-30,notebook,37.53,1,98101
4,1114,2026-04-06,webcam,45.59,4,90405
5,1280,2026-04-22,pen set,37.53,4,30303
6,1204,2026-04-18,charger,8.41,2,02134
7,1249,2026-05-15,keyboard,75.04,4,90405
8,1037,2026-05-19,mug,9.31,2,60614
9,1177,2026-05-13,mug,10.11,3,90210


In [12]:
df.dtypes

order_id             int64
date        datetime64[us]
product                str
price              float64
qty                  int64
zip                    str
dtype: object

In [13]:
df.shape

(292, 6)

In [14]:
## checking for negative values in qty
df[df["qty"]<0]

,order_id,date,product,price,qty,zip
202,1140,2026-04-18,webcam,38.69,-5,98101
262,1233,2026-05-09,desk lamp,22.64,-4,10001
297,1025,2026-04-27,keyboard,47.30,-2,02116


In [15]:
## removing rows with negative quantity values
df=df[df["qty"]>0]

In [16]:
## verifying the rows were removed
df.shape

(289, 6)

In [17]:
df.describe()

,order_id,date,price,qty
count,289.000000,289,289.000000,289.000000
mean,1145.633218,2026-05-01 17:46:17.854671,55.723633,2.979239
min,1000.000000,2026-04-01 00:00:00,2.930000,1.000000
25%,1073.000000,2026-04-18 00:00:00,10.970000,2.000000
50%,1146.000000,2026-05-02 00:00:00,37.530000,3.000000
75%,1218.000000,2026-05-15 00:00:00,56.380000,4.000000
max,1291.000000,2026-05-30 00:00:00,479.290000,5.000000
std,84.420433,NaN,72.105147,1.409141


In [18]:
df.to_csv("sales_clean.csv", index=False)

## Cleaning Log
- 300 loaded
- 12 prices filled with 37.53
- 8 duplicates removed leaving 292 rows
- zips restored
- dates standarsized 
- negatives removed leaving 289 rows because there was no sure way to no if they were typos or returns. Assuming they were typos or returns and being wrong could inflate or negatively affect the sales numbers, so I decided on dropping them as it didn't affect a large percentage of the data and it would have the least affect on the data.

### How the wrong choice can affect the business decision.
If the business continues to assume that all negatives are typos, when really they were returns, a business might see, webcams for example, as an extremely high seller, and continue to buy more even though the reality is most are being returned. The business would continue wasting money on products that always end up being returned, their inventory will be off, and they will also be wasting warehouse space, where if they were to recognize the returns, they would probably stop selling the product all together.